In [21]:
from google.colab import files

# Ouvre une fenêtre pour sélectionner un fichier
uploaded = files.upload()

Saving louvre_entities.csv to louvre_entities (1).csv
Saving top_20_entities.csv to top_20_entities (2).csv


In [22]:
import pandas as pd
import networkx as nx
import json
import re
from collections import defaultdict

# ─────────────────────────────────────────────
# 1. Loading entities
# ─────────────────────────────────────────────

def load_entities(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Normalisation
    df["entity_clean"] = df["entity"].str.strip().str.title()
    df["entity_type"]  = df["entity_type"].str.strip().str.upper()

    # Remove parasitic entities
    noise_pattern = r"^\d{4}(-\d{2}-\d{2})?$|^(pp|edit|https?://|--|issn).*"
    df = df[~df["entity_clean"].str.lower().str.match(noise_pattern, na=False)]
    df = df[df["entity_clean"].str.len() > 3]

    return df


# ─────────────────────────────────────────────
# 2. Definition of key manual relationships
# ─────────────────────────────────────────────

MANUAL_RELATIONS = [
    # (sujet, relation, objet)
    ("Mona Lisa",         "CREATED_BY",      "Leonardo Da Vinci"),
    ("Mona Lisa",         "LOCATED_IN",      "Louvre Museum"),
    ("Mona Lisa",         "DEPICTS",         "Lisa Del Giocondo"),
    ("Mona Lisa",         "MEDIUM",          "Oil On Poplar Panel"),
    ("Venus De Milo",     "LOCATED_IN",      "Louvre Museum"),
    ("Venus De Milo",     "ORIGIN",          "Greece"),
    ("Winged Victory",    "LOCATED_IN",      "Louvre Museum"),
    ("Winged Victory",    "ORIGIN",          "Greece"),
    ("Louvre Museum",     "LOCATED_IN",      "Paris"),
    ("Louvre Museum",     "LOCATED_IN",      "France"),
    ("Louvre Museum",     "FOUNDED_BY",      "Napoleon"),
    ("Leonardo Da Vinci", "NATIONALITY",     "Italy"),
    ("Leonardo Da Vinci", "BORN_IN",         "Vinci"),
    ("Michelangelo",      "NATIONALITY",     "Italy"),
    ("Delacroix",         "NATIONALITY",     "France"),
    ("Francis I",         "ACQUIRED",        "Mona Lisa"),
    ("Francis I",         "NATIONALITY",     "France"),
]


def infer_relations_from_context(df):
    """
    Infers additional relationships by detecting the co-occurrence
of entities in the same textual context.
    """
    relations = []
    for _, row in df.iterrows():
        ctx = row.get("context", "")
        if not isinstance(ctx, str):
            continue
        # Search for other entities mentioned in the same context
        for _, other in df[df["source_file"] == row["source_file"]].iterrows():
            if other["entity_clean"] == row["entity_clean"]:
                continue
            if other["entity_clean"] in ctx:
                if row["entity_type"] == "WORK_OF_ART" and other["entity_type"] == "PERSON":
                    relations.append((row["entity_clean"], "RELATED_TO_ARTIST", other["entity_clean"]))
                elif row["entity_type"] in ("ORG", "GPE") and other["entity_type"] == "WORK_OF_ART":
                    relations.append((other["entity_clean"], "ASSOCIATED_WITH", row["entity_clean"]))
    return relations[:200]   # limit to avoid combinatorial explosion


# ─────────────────────────────────────────────
# 3. Construction of the graph
# ─────────────────────────────────────────────

TYPE_COLORS = {
    "PERSON":      "#4A90D9",
    "ORG":         "#E67E22",
    "GPE":         "#27AE60",
    "WORK_OF_ART": "#8E44AD",
    "DATE":        "#95A5A6",
    "UNKNOWN":     "#BDC3C7",
}

def build_graph(df: pd.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()

    # Adding nodes (unique entities)
    unique_entities = df.groupby("entity_clean")["entity_type"].agg(
        lambda x: x.mode()[0]
    ).reset_index()

    for _, row in unique_entities.iterrows():
        G.add_node(
            row["entity_clean"],
            entity_type=row["entity_type"],
            color=TYPE_COLORS.get(row["entity_type"], TYPE_COLORS["UNKNOWN"]),
        )

    # Adding manual relationships
    for subj, rel, obj in MANUAL_RELATIONS:
        if subj not in G:
            G.add_node(subj, entity_type="UNKNOWN", color=TYPE_COLORS["UNKNOWN"])
        if obj not in G:
            G.add_node(obj,  entity_type="UNKNOWN", color=TYPE_COLORS["UNKNOWN"])
        G.add_edge(subj, obj, relation=rel)

    # Adding inferred relationships
    inferred = infer_relations_from_context(df)
    for subj, rel, obj in inferred:
        if subj in G and obj in G:
            if not G.has_edge(subj, obj):
                G.add_edge(subj, obj, relation=rel)

    return G


# ─────────────────────────────────────────────
# 4. Exporting the graph
# ─────────────────────────────────────────────

def export_graph(G: nx.DiGraph, out_dir: str = "."):
    #  GraphML
    nx.write_graphml(G, f"{out_dir}/louvre_knowledge_graph.graphml")
    print(f"GraphML exported → {out_dir}/louvre_knowledge_graph.graphml")

    # ── Statistiques ──
    print(f"\n Graph statistics :")
    print(f"   Nodes  : {G.number_of_nodes()}")
    print(f"   Edges : {G.number_of_edges()}")
    by_type = defaultdict(int)
    for n in G.nodes:
        by_type[G.nodes[n].get("entity_type","?")] += 1
    for t, c in sorted(by_type.items()):
        print(f"   {t:15s}: {c}")


# ─────────────────────────────────────────────
# 5. Utility queries on the graph
# ─────────────────────────────────────────────

def query_neighbors(G, entity, direction = "both"):
    """
    Returns the neighbors of an entity along with their relationships.
Direction: 'out' (outgoing), 'in' (incoming), 'both'
    """
    result = {"entity": entity, "relations": []}
    if entity not in G:
        return {"error": f"Entité '{entity}' non trouvée dans le graphe."}

    if direction in ("out", "both"):
        for _, target, data in G.out_edges(entity, data=True):
            result["relations"].append({
                "direction": "→",
                "relation":  data.get("relation", "RELATED_TO"),
                "target":    target,
                "target_type": G.nodes[target].get("entity_type", "?"),
            })
    if direction in ("in", "both"):
        for source, _, data in G.in_edges(entity, data=True):
            result["relations"].append({
                "direction": "←",
                "relation":  data.get("relation", "RELATED_TO"),
                "target":    source,
                "target_type": G.nodes[source].get("entity_type", "?"),
            })
    return result


def find_path(G: nx.DiGraph, source: str, target: str) -> list:
    """Find the shortest path between two entities."""
    try:
        path = nx.shortest_path(G.to_undirected(), source, target)
        return path
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return []


# ─────────────────────────────────────────────
# 6. Main
# ─────────────────────────────────────────────

if __name__ == "__main__":
    print(" Loading entities...")
    df = load_entities("louvre_entities.csv")
    print(f"   {len(df)} charged entities ({df['entity_clean'].nunique()} uniques)")

    print("\n Construction of the graph...")
    G = build_graph(df)

    print("\n Export...")
    import os
    os.makedirs("output", exist_ok=True)
    export_graph(G, out_dir="output")

    # Example queries
    print("\n Example query — neighbors of 'Mona Lisa' :")
    result = query_neighbors(G, "Mona Lisa")
    for r in result["relations"]:
        print(f"   Mona Lisa {r['direction']} [{r['relation']}] {r['target']} ({r['target_type']})")

    print("\n The path between 'Leonardo Da Vinci' and 'France' :")
    path = find_path(G, "Leonardo Da Vinci", "France")
    print("   " + " → ".join(path) if path else "   No path found")

 Loading entities...
   2868 charged entities (2707 uniques)

 Construction of the graph...

 Export...
GraphML exported → output/louvre_knowledge_graph.graphml

 Graph statistics :
   Nodes  : 2710
   Edges : 217
   DATE           : 685
   GPE            : 257
   ORG            : 764
   PERSON         : 891
   UNKNOWN        : 3
   WORK_OF_ART    : 110

 Example query — neighbors of 'Mona Lisa' :
   Mona Lisa → [CREATED_BY] Leonardo Da Vinci (PERSON)
   Mona Lisa → [LOCATED_IN] Louvre Museum (ORG)
   Mona Lisa → [DEPICTS] Lisa Del Giocondo (PERSON)
   Mona Lisa → [MEDIUM] Oil On Poplar Panel (UNKNOWN)
   Mona Lisa ← [ACQUIRED] Francis I (UNKNOWN)
   Mona Lisa ← [RELATED_TO_ARTIST] The Theft Of The Mona Lisa (WORK_OF_ART)
   Mona Lisa ← [RELATED_TO_ARTIST] The Baptism Of Christ (WORK_OF_ART)
   Mona Lisa ← [RELATED_TO_ARTIST] La Joconde À La Cuiller (WORK_OF_ART)
   Mona Lisa ← [RELATED_TO_ARTIST] Mona Lisa Lost Her Smile (WORK_OF_ART)
   Mona Lisa ← [RELATED_TO_ARTIST] "The Ballad Of 

In [23]:
"""
RAG System (Retrieval-Augmented Generation)

"""

import os
import json
import math
import re
from pathlib import Path
from collections import defaultdict

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

CLEANED_DATA_DIR  = "data/cleaned"
GRAPH_JSON_PATH   = "output/louvre_knowledge_graph.json"
TOP_K_CHUNKS      = 4
CHUNK_SIZE        = 300
CHUNK_OVERLAP     = 50
MAX_TOKENS        = 800                     # ignored here, for compatibility


# ─────────────────────────────────────────────
# 1. CORPUS LOADING
# ─────────────────────────────────────────────

def load_corpus(data_dir: str):
    chunks = []
    data_path = Path(data_dir)
    if not data_path.exists():
        print(f"Folder '{data_dir}' not found. Using demo corpus.")
        return _demo_corpus()
    files = list(data_path.glob("*.txt"))
    if not files:
        print(f"No .txt files in '{data_dir}'. Using demo corpus.")
        return _demo_corpus()
    for filepath in files:
        text = filepath.read_text(encoding="utf-8", errors="ignore")
        words = text.split()
        step = CHUNK_SIZE - CHUNK_OVERLAP
        for i in range(0, max(1, len(words) - CHUNK_OVERLAP), step):
            chunk_words = words[i : i + CHUNK_SIZE]
            if len(chunk_words) < 20:
                continue
            chunks.append({
                "text": " ".join(chunk_words),
                "source": filepath.name,
                "chunk_id": f"{filepath.stem}_{i // step}",
            })
    print(f"Corpus loaded: {len(chunks)} chunks from {len(files)} files")
    return chunks


def _demo_corpus():
    texts = [
        ("The Mona Lisa is a half-length portrait painting by Italian artist Leonardo da Vinci. "
         "Considered an archetypal masterpiece of the Italian Renaissance, it has been described as "
         "the best known, the most visited, the most written about, the most sung about, the most "
         "parodied work of art in the world. The painting's novel qualities include the subject's "
         "expression, which is frequently described as enigmatic, the monumentality of the composition, "
         "the subtle modelling of forms, and the atmospheric illusionism. The work is owned by the "
         "French Republic and has been on permanent display at the Louvre Museum in Paris since 1797.",
         "mona_lisa_demo"),
        ("The Venus de Milo is an ancient Greek sculpture and one of the most famous works of ancient "
         "Greek sculpture. Created sometime between 130 and 100 BC, it is believed to depict Aphrodite, "
         "the Greek goddess of love and beauty. It is a marble sculpture, slightly larger than life size "
         "at 2.02m high. It is currently on display at the Louvre Museum in Paris.",
         "venus_de_milo_demo"),
        ("The Louvre or the Louvre Museum is a national art museum in Paris, France. A central landmark "
         "of the city, it is located on the Right Bank of the Seine in the city's 1st arrondissement. "
         "The museum opened on 10 August 1793. It receives around 9 million visitors per year. "
         "The Louvre Palace was built in the late 12th to 13th century under Philip II. Remains of the "
         "medieval Louvre fortress and moat can be seen in the basement of the museum. "
         "Napoleon Bonaparte oversaw a major renovation of the museum.",
         "louvre_museum_demo"),
        ("The Winged Victory of Samothrace, also called the Nike of Samothrace, is a marble Hellenistic "
         "sculpture of Nike, the goddess of victory. Since 1884 it has been prominently displayed at the "
         "Louvre Museum and is considered one of the greatest masterpieces of sculpture. "
         "The goddess is depicted as if landing on the prow of a ship, with her wings spread. "
         "It was created about 200–190 BC and stands 2.75 metres tall.",
         "winged_victory_demo"),
    ]
    chunks = []
    for text, source in texts:
        chunks.append({"text": text, "source": source, "chunk_id": source + "_0"})
    print(f"Demo corpus loaded: {len(chunks)} chunks")
    return chunks


# ─────────────────────────────────────────────
# 2. TF-IDF INDEX
# ─────────────────────────────────────────────

def _tokenize(text: str):
    return re.findall(r"[a-z']+", text.lower())

def build_tfidf_index(chunks):
    N = len(chunks)
    df_counts = defaultdict(int)
    tf_matrices = []
    for chunk in chunks:
        tokens = _tokenize(chunk["text"])
        tf = defaultdict(int)
        for t in tokens:
            tf[t] += 1
        total = max(len(tokens), 1)
        tf_norm = {t: c / total for t, c in tf.items()}
        tf_matrices.append(tf_norm)
        for t in tf:
            df_counts[t] += 1
    idf = {t: math.log((N + 1) / (df + 1)) + 1 for t, df in df_counts.items()}
    tfidf_vectors = [{t: tf_norm[t] * idf.get(t, 1) for t in tf_norm} for tf_norm in tf_matrices]
    return {"idf": idf, "vectors": tfidf_vectors, "chunks": chunks}

def retrieve(query, index, top_k=TOP_K_CHUNKS):
    idf     = index["idf"]
    vectors = index["vectors"]
    chunks  = index["chunks"]
    q_tokens = _tokenize(query)
    q_tf = defaultdict(int)
    for t in q_tokens:
        q_tf[t] += 1
    total = max(len(q_tokens), 1)
    q_vec = {t: (c / total) * idf.get(t, 1) for t, c in q_tf.items()}

    def cosine(a, b):
        common = set(a) & set(b)
        if not common:
            return 0.0
        dot = sum(a[t] * b[t] for t in common)
        norm_a = math.sqrt(sum(v ** 2 for v in a.values()))
        norm_b = math.sqrt(sum(v ** 2 for v in b.values()))
        return dot / (norm_a * norm_b + 1e-9)

    scores = [(cosine(q_vec, vec), i) for i, vec in enumerate(vectors)]
    scores.sort(reverse=True)
    return [{**chunks[idx], "score": round(score, 4)} for score, idx in scores[:top_k]]


# ─────────────────────────────────────────────
# 3. GRAPH ENRICHMENT
# ─────────────────────────────────────────────

def load_graph(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Graph not found ({path}). Using embedded demo graph.")
        return _demo_graph()

def _demo_graph():
    nodes = [
        {"id": "Mona Lisa",         "entity_type": "WORK_OF_ART"},
        {"id": "Leonardo Da Vinci", "entity_type": "PERSON"},
        {"id": "Louvre Museum",     "entity_type": "ORG"},
        {"id": "Paris",             "entity_type": "GPE"},
        {"id": "France",            "entity_type": "GPE"},
        {"id": "Venus De Milo",     "entity_type": "WORK_OF_ART"},
        {"id": "Winged Victory",    "entity_type": "WORK_OF_ART"},
        {"id": "Napoleon",          "entity_type": "PERSON"},
        {"id": "Greece",            "entity_type": "GPE"},
        {"id": "Lisa Del Giocondo", "entity_type": "PERSON"},
    ]
    edges = [
        {"source": "Mona Lisa",      "target": "Leonardo Da Vinci", "relation": "CREATED_BY"},
        {"source": "Mona Lisa",      "target": "Louvre Museum",     "relation": "LOCATED_IN"},
        {"source": "Mona Lisa",      "target": "Lisa Del Giocondo", "relation": "DEPICTS"},
        {"source": "Venus De Milo",  "target": "Louvre Museum",     "relation": "LOCATED_IN"},
        {"source": "Venus De Milo",  "target": "Greece",            "relation": "ORIGIN"},
        {"source": "Winged Victory", "target": "Louvre Museum",     "relation": "LOCATED_IN"},
        {"source": "Louvre Museum",  "target": "Paris",             "relation": "LOCATED_IN"},
        {"source": "Louvre Museum",  "target": "France",            "relation": "LOCATED_IN"},
        {"source": "Louvre Museum",  "target": "Napoleon",          "relation": "FOUNDED_BY"},
    ]
    return {"nodes": nodes, "edges": edges}

def get_graph_context(query, graph, max_facts=10):
    q_words = set(_tokenize(query))
    node_ids = {n["id"] for n in graph["nodes"]}
    matched_nodes = {nid for nid in node_ids if set(_tokenize(nid)) & q_words}
    if not matched_nodes:
        matched_nodes = {"Mona Lisa", "Louvre Museum", "Leonardo Da Vinci"}
    facts = []
    for edge in graph["edges"]:
        if edge["source"] in matched_nodes or edge["target"] in matched_nodes:
            facts.append(f"{edge['source']} --[{edge['relation']}]--> {edge['target']}")
        if len(facts) >= max_facts:
            break
    return "\n".join(facts) if facts else "No facts found in the graph."


# ─────────────────────────────────────────────
# 4. SIMULATED GENERATION
# ─────────────────────────────────────────────

def generate_answer(query, passages, graph_context):
    """Simulates Claude-like generation without API."""
    if not passages:
        return "No information found to answer the question."
    best_passage = passages[0]
    return (
        f"Simulated answer based on the most relevant passage:\n\n"
        f"{best_passage['text']}\n\n"
        f"Graph context:\n{graph_context}"
    )


# ─────────────────────────────────────────────
# 5. RAG PIPELINE
# ─────────────────────────────────────────────

class LouvreRAG:
    def __init__(self):
        print("Initializing Louvre RAG system...\n")
        self.chunks = load_corpus(CLEANED_DATA_DIR)
        self.index  = build_tfidf_index(self.chunks)
        self.graph  = load_graph(GRAPH_JSON_PATH)
        print("\nSystem ready.\n")

    def ask(self, query, verbose=True):
        passages = retrieve(query, self.index, top_k=TOP_K_CHUNKS)
        if verbose:
            print(f"\nRetrieved passages ({len(passages)}):")
            for p in passages:
                print(f"   [{p['score']:.3f}] {p['source']} : {p['text'][:100]}...")
        graph_ctx = get_graph_context(query, self.graph)
        if verbose:
            print(f"\nGraph context:\n{graph_ctx}\n")
        answer = generate_answer(query, passages, graph_ctx)
        return answer


# ─────────────────────────────────────────────
# EXAMPLE USAGE IN COLAB/JUPYTER
# ─────────────────────────────────────────────

rag = LouvreRAG()
response = rag.ask("Who painted the Mona Lisa?", verbose=True)
print(f"\nFinal answer:\n{response}")

Initializing Louvre RAG system...

Folder 'data/cleaned' not found. Using demo corpus.
Demo corpus loaded: 4 chunks

System ready.


Retrieved passages (4):
   [0.303] mona_lisa_demo : The Mona Lisa is a half-length portrait painting by Italian artist Leonardo da Vinci. Considered an ...
   [0.190] louvre_museum_demo : The Louvre or the Louvre Museum is a national art museum in Paris, France. A central landmark of the...
   [0.129] winged_victory_demo : The Winged Victory of Samothrace, also called the Nike of Samothrace, is a marble Hellenistic sculpt...
   [0.081] venus_de_milo_demo : The Venus de Milo is an ancient Greek sculpture and one of the most famous works of ancient Greek sc...

Graph context:
"The Ballad Of Mona Lisa --[RELATED_TO_ARTIST]--> Mona Lisa
"The Ballad Of Mona Lisa --[RELATED_TO_ARTIST]--> Lisa
"The Ballad Of Mona Lisa --[RELATED_TO_ARTIST]--> David Allan Coe
"The Ballad Of Mona Lisa --[RELATED_TO_ARTIST]--> Disco
"The Ballad Of Mona Lisa --[RELATED_TO_ARTIST]-->